<a href="https://colab.research.google.com/github/naisyahnurazizah11-ai/naisyah/blob/main/regresi_naisyah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.api as sms
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import plotly.graph_objects as go
import plotly.io as pio

# Memaksa Plotly untuk merender grafik interaktif langsung di dalam sel Google Colab
pio.renderers.default = 'colab'

# 1. MEMBACA DATA ASLI DARI GITHUB (KOTA MALANG)

url = 'https://raw.githubusercontent.com/Alam-A99/DATAAKSI/main/dataset_fiskal_97_hipoFull.csv'
df_all = pd.read_csv(url)

# Menentukan kolom wilayah dan menyaring khusus Kota Malang
kolom_kota = 'Kota' if 'Kota' in df_all.columns else [c for c in df_all.columns if 'kota' in c.lower() or 'kab' in c.lower()][0]

# Memfilter Kota Malang (serta memastikan Kabupaten Malang tidak ikut terambil)
df = df_all[df_all[kolom_kota].astype(str).str.contains('Malang', case=False, na=False)].copy()
if df[kolom_kota].str.contains('Kabupaten', case=False).any():
    df = df[df[kolom_kota].astype(str).str.contains('Kota Malang', case=False, na=False)].copy()

# Menampilkan data Kota Malang
print("Data Kota Malang:")
display(df)
print(f"Total data Kota Malang yang dianalisis: {len(df)} baris.\n")

Data Kota Malang:


,Kota,Tahun,DBH_PAJAK,DBH_SDA,DBH_Lainnya,DAU_Grant,DAU_Earmark,DAK_Fisik,DAK_Nonfisik,PAD,...,bm_lingkungan,bm_ekonomi,bm_infrastruktur,bm_pariwisata,bm_perlindungan_sosial,KEMISKINAN,PENGANGGURAN,KETIMPANGAN,IPM,PE
561,Kota Malang,2014,146.23,97.57,85.76,295.54,133.60,104.74,150.38,310.45,...,182.86,175.84,278.71,210.53,138.37,7.53,8.32,0.50,83.64,2.52
562,Kota Malang,2015,194.48,124.91,75.38,263.47,191.81,153.49,142.40,309.70,...,121.99,249.11,133.85,124.54,210.55,13.69,4.32,0.48,76.85,7.62
563,Kota Malang,2016,123.32,123.84,62.86,233.15,171.30,157.34,178.20,244.34,...,173.20,234.87,288.07,81.62,111.41,11.46,4.22,0.50,74.03,4.27
564,Kota Malang,2017,157.68,70.76,117.53,271.99,169.39,196.28,121.91,430.73,...,124.99,211.38,183.26,260.64,77.32,13.05,4.95,0.31,64.06,5.46
565,Kota Malang,2018,219.41,97.59,69.33,350.14,204.62,182.77,177.45,316.69,...,199.02,88.94,207.89,140.55,248.60,15.04,9.79,0.34,74.93,6.75
566,Kota Malang,2019,178.44,93.02,112.56,306.54,190.37,160.93,145.77,267.88,...,158.46,186.76,185.74,215.12,74.74,15.71,4.37,0.44,69.76,3.95
567,Kota Malang,2020,187.48,110.73,70.22,294.72,212.30,212.57,164.13,372.64,...,91.10,71.86,82.35,167.70,117.85,7.90,4.47,0.30,82.42,6.30
568,Kota Malang,2021,151.56,96.26,69.60,325.86,132.75,140.04,203.07,330.81,...,79.15,144.34,143.96,99.73,263.78,15.62,6.90,0.35,76.27,2.81
569,Kota Malang,2022,187.74,114.51,77.46,318.19,228.66,244.25,196.46,343.29,...,186.67,153.38,74.79,200.42,136.62,8.38,4.80,0.40,69.18,3.75
570,Kota Malang,2023,230.13,146.90,116.35,477.08,277.64,271.48,224.39,611.50,...,275.60,59.95,205.74,65.17,293.26,5.58,9.10,0.40,69.96,2.87


Total data Kota Malang yang dianalisis: 11 baris.



In [ ]:
# 2. MODEL REGRESI & UJI ASUMSI KLASIK

X = df[['bm_pendidikan', 'bm_perlindungan_sosial']]
Y = df['KEMISKINAN']
X_sm = sm.add_constant(X)
model = sm.OLS(Y, X_sm).fit()

# Menampilkan Summary OLS
print(model.summary())

print("\n" + "="*50)
print("FUNGSI REGRESI (Data Asli):")
print(f"Y = {model.params['const']:.4f} + {model.params['bm_pendidikan']:.4f}(BM_Pendidikan) - {abs(model.params['bm_perlindungan_sosial']):.4f}(BM_Sosial)")
print("="*50)

print("\nHASIL UJI ASUMSI KLASIK")
# 1. Normalitas
shapiro_stat, shapiro_p = stats.shapiro(model.resid)
print(f"- Normalitas (Shapiro-Wilk) P-Value: {shapiro_p:.4f} (>0.05 = Lolos)")

# 2. Multikolinearitas (VIF)
vif_data = pd.DataFrame()
vif_data["Variabel"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(f"- Multikolinearitas (VIF)             : {vif_data['VIF'].max():.4f} (<10 = Lolos)")

# 3. Heteroskedastisitas
bp_test = sms.het_breuschpagan(model.resid, model.model.exog)
print(f"- Heteroskedastisitas (BP) P-Value   : {bp_test[1]:.4f} (>0.05 = Lolos)")

# 4. Autokorelasi
dw_value = sm.stats.stattools.durbin_watson(model.resid)
print(f"- Autokorelasi (Durbin-Watson)       : {dw_value:.4f} (Mendekati 2 = Lolos)\n")

                            OLS Regression Results                            
Dep. Variable:             KEMISKINAN   R-squared:                       0.104
Model:                            OLS   Adj. R-squared:                 -0.120
Method:                 Least Squares   F-statistic:                    0.4657
Date:                Wed, 14 Oct 2026   Prob (F-statistic):              0.644
Time:                        22:09:51   Log-Likelihood:                -28.497
No. Observations:                  11   AIC:                             62.99
Df Residuals:                       8   BIC:                             64.19
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     10

In [ ]:
# --------------------------------------------------------------------------------
# 3. VISUALISASI 3D INTERAKTIF

# Membuat grid bidang regresi berdasarkan rentang data asli
x_range = np.linspace(df['bm_pendidikan'].min(), df['bm_pendidikan'].max(), 20)
y_range = np.linspace(df['bm_perlindungan_sosial'].min(), df['bm_perlindungan_sosial'].max(), 20)
x_grid, y_grid = np.meshgrid(x_range, y_range)

# Menghitung prediksi Z untuk permukaan bidang
z_grid = model.params['const'] + (model.params['bm_pendidikan'] * x_grid) + (model.params['bm_perlindungan_sosial'] * y_grid)

# Inisialisasi Plotly Figure
fig = go.Figure()

# Menambahkan Bidang Regresi Permukaan
fig.add_trace(go.Surface(
    x=x_grid,
    y=y_grid,
    z=z_grid,
    name='Bidang Regresi',
    colorscale='viridis',
    opacity=0.8,
    showscale=False
))

# Menambahkan Titik Data Aktual Asli
fig.add_trace(go.Scatter3d(
    x=df['bm_pendidikan'],
    y=df['bm_perlindungan_sosial'],
    z=df['KEMISKINAN'],
    mode='markers',
    name='Data Aktual',
    marker=dict(size=6, color='red', line=dict(color='white', width=1))
))

# Mengatur Tampilan Layout Grafik
fig.update_layout(
    title=f"Plot Bidang Regresi 3D: Kemiskinan terhadap BM Pendidikan & BM Sosial (Kota Malang) | R² = {model.rsquared:.2f}",
    template='plotly_dark',
    scene=dict(
        xaxis_title='BM Pendidikan',
        yaxis_title='BM Perlindungan Sosial',
        zaxis_title='Kemiskinan'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# Menampilkan kanvas interaktif di bawah sel Colab
fig.show()